# 08 - Graficas finales

Este notebook contiene explicitamente las funciones que generan las figuras finales. Todas las entradas se leen desde `data/`, `results/tables/` y `results/predictions/` dentro de esta carpeta oficial.


## Definicion de funciones graficas

Esta celda contiene todas las funciones que leen datos, predicciones y tablas para generar las figuras finales.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats


ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
PREDICTIONS = RESULTS / "predictions"
TABLES = RESULTS / "tables"
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

DATA_START = "2010-01-01"
EVAL_START = "2015-01-02"
EVAL_END = "2024-12-27"
MLP_LABEL = "MLP-VaR"
MLP_LONG_LABEL = "MLP Softplus SiLU LayerNorm downside pressure"


def savefig(name: str) -> None:
    plt.tight_layout()
    plt.savefig(FIGURES / f"{name}.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIGURES / f"{name}.pdf", bbox_inches="tight")
    plt.savefig(FIGURES / f"{name}.jpg", dpi=300, bbox_inches="tight")
    plt.close()


def load_dataset() -> pd.DataFrame:
    return pd.read_pickle(DATA / "dataset_tfg.pkl").sort_index()


def load_predictions() -> dict[str, tuple[pd.DataFrame, str]]:
    return {
        MLP_LABEL: (pd.read_csv(PREDICTIONS / "mlp_var_predictions.csv", index_col=0, parse_dates=True), "VaR_pred"),
        "GARCH-t": (pd.read_csv(PREDICTIONS / "garch_t_var_predictions.csv", index_col=0, parse_dates=True), "VaR_GARCH"),
        "GARCH-Normal": (pd.read_csv(PREDICTIONS / "garch_n_var_predictions.csv", index_col=0, parse_dates=True), "VaR_GARCH_n"),
        "HS": (pd.read_csv(PREDICTIONS / "hs_var_predictions.csv", index_col=0, parse_dates=True), "VaR_HS"),
        "CAViaR-AS": (pd.read_csv(PREDICTIONS / "caviar_var_predictions.csv", index_col=0, parse_dates=True), "VaR_CAViaR"),
    }


def plot_returns_time_series() -> None:
    df = load_dataset().loc[DATA_START:EVAL_END]
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(df.index, df["rp_lag_0"], lw=0.7, color="#1f4e79")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title("Rentabilidad diaria de la cartera (2010-2024)")
    ax.set_ylabel("Rentabilidad")
    savefig("01_returns_time_series")


def plot_loss_histogram() -> None:
    df = load_dataset().loc[EVAL_START:EVAL_END]
    losses = df["target"].dropna()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(losses, bins=60, color="#6b8e23", edgecolor="white")
    ax.set_title("Distribucion de perdidas de la cartera (2015-2024)")
    ax.set_xlabel("Perdida")
    ax.set_ylabel("Frecuencia")
    savefig("02_loss_histogram")


def plot_qq_normal() -> None:
    df = load_dataset().loc[EVAL_START:EVAL_END]
    losses = df["target"].dropna()

    fig = plt.figure(figsize=(5, 5))
    stats.probplot(losses, dist="norm", plot=plt)
    plt.title("QQ plot frente a normal (2015-2024)")
    savefig("03_qq_normal")


def plot_histogram_and_qq() -> None:
    plot_loss_histogram()
    plot_qq_normal()


def plot_mlp_var_full() -> None:
    df = pd.read_csv(PREDICTIONS / "mlp_var_predictions.csv", index_col=0, parse_dates=True)
    df = df.loc[EVAL_START:EVAL_END]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(df.index, df["loss_real"], lw=0.65, color="#333333", label="Perdida realizada")
    ax.plot(df.index, df["VaR_pred"], lw=1.0, color="#b22222", label=MLP_LABEL)
    viol = df[df["viol"] == 1]
    ax.scatter(viol.index, viol["loss_real"], s=16, color="#ff7f0e", label="Violaciones", zorder=3)
    ax.set_title("MLP-VaR: pérdidas realizadas y estimaciones del VaR99% (2015-2024)")
    ax.set_ylabel("Perdida / VaR")
    ax.legend(loc="upper left")
    savefig("04_mlp_var_full")


def plot_covid_zoom() -> None:
    df = pd.read_csv(PREDICTIONS / "mlp_var_predictions.csv", index_col=0, parse_dates=True)
    z = df.loc["2020-01-01":"2020-12-31"]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(z.index, z["loss_real"], lw=0.8, color="#333333", label="Perdida realizada")
    ax.plot(z.index, z["VaR_pred"], lw=1.2, color="#b22222", label=MLP_LABEL)
    viol = z[z["viol"] == 1]
    ax.scatter(viol.index, viol["loss_real"], s=20, color="#ff7f0e", label="Violaciones", zorder=3)
    ax.set_title("Zoom 2020: VaR MLP y shock COVID")
    ax.legend(loc="upper left")
    savefig("05_zoom_covid_mlp")


def plot_benchmark_panel() -> None:
    preds = load_predictions()
    common_idx = None
    for df, _ in preds.values():
        common_idx = df.index if common_idx is None else common_idx.intersection(df.index)
    common_idx = common_idx.sort_values()

    fig, axes = plt.subplots(5, 1, figsize=(11, 11), sharex=True)
    for ax, (name, (df, var_col)) in zip(axes, preds.items()):
        aligned = df.loc[common_idx]
        ax.plot(aligned.index, aligned["loss_real"], lw=0.45, color="#444444")
        ax.plot(aligned.index, aligned[var_col], lw=0.9, color="#b22222")
        ax.set_title(name, loc="left", fontsize=10)
        ax.set_ylabel("VaR")
    axes[-1].set_xlabel("Fecha")
    fig.suptitle("Comparacion temporal de VaR: MLP y benchmarks", y=0.995)
    savefig("06_panel_benchmarks")


def plot_violations_by_year() -> None:
    yearly = pd.read_csv(TABLES / "model_yearly_comparison_2015_2024.csv")
    yearly["Model"] = yearly["Model"].replace({MLP_LONG_LABEL: MLP_LABEL})
    pivot = yearly.pivot(index="year", columns="Model", values="Violations")
    fig, ax = plt.subplots(figsize=(11, 4.5))
    pivot.plot(kind="bar", ax=ax)
    ax.set_title("Violaciones anuales por modelo")
    ax.set_ylabel("Numero de violaciones")
    ax.axhline(2.5, color="black", lw=0.8, ls="--", label="Esperadas aprox.")
    ax.legend(fontsize=8)
    savefig("07_violations_by_year")


def plot_temporal_violations_valid_models() -> None:
    preds = load_predictions()
    valid_models = ["MLP-VaR", "GARCH-t", "CAViaR-AS"]
    colors = {"MLP-VaR": "#9467bd", "GARCH-t": "#2ca02c", "CAViaR-AS": "#1f77b4"}

    fig, ax = plt.subplots(figsize=(11, 3.8))
    y_positions = {model: i for i, model in enumerate(valid_models)}

    for model in valid_models:
        df, var_col = preds[model]
        df = df.loc[EVAL_START:EVAL_END].copy()
        if "viol" not in df.columns:
            df["viol"] = (df["loss_real"] > df[var_col]).astype(int)
        violations = df[df["viol"] == 1]
        total = int(df["viol"].sum())
        rate = total / len(df)
        y = np.full(len(violations), y_positions[model])
        ax.scatter(
            violations.index,
            y,
            s=34,
            color=colors[model],
            label=f"{model}: {total} viol. ({rate:.2%})",
        )

    ax.set_yticks(list(y_positions.values()))
    ax.set_yticklabels(valid_models)
    ax.set_ylim(-0.6, len(valid_models) - 0.4)
    ax.set_xlim(pd.Timestamp(EVAL_START), pd.Timestamp(EVAL_END))
    ax.grid(axis="x", alpha=0.25)
    ax.set_xlabel("Fecha")
    ax.set_title("Distribución temporal de violaciones del VaR99% (2015-2024): modelos válidos")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, fontsize=9)
    savefig("09_temporal_violations_valid_models")


def plot_violation_rate_lines() -> None:
    yearly = pd.read_csv(TABLES / "model_yearly_comparison_2015_2024.csv")
    yearly["Model"] = yearly["Model"].replace({MLP_LONG_LABEL: MLP_LABEL})

    order = ["MLP-VaR", "GARCH-t", "GARCH-Normal", "Historical Simulation", "CAViaR-AS"]
    labels = {"Historical Simulation": "HS"}
    colors = {
        "MLP-VaR": "#9467bd",
        "GARCH-t": "#2ca02c",
        "GARCH-Normal": "#ff7f0e",
        "Historical Simulation": "#d62728",
        "CAViaR-AS": "#1f77b4",
    }
    markers = {
        "MLP-VaR": "o",
        "GARCH-t": "s",
        "GARCH-Normal": "^",
        "Historical Simulation": "D",
        "CAViaR-AS": "P",
    }

    fig, ax = plt.subplots(figsize=(11, 4.8))
    for model in order:
        series = yearly[yearly["Model"] == model].sort_values("year")
        ax.plot(
            series["year"],
            series["Violation Rate"],
            marker=markers[model],
            lw=1.8,
            color=colors[model],
            label=labels.get(model, model),
        )

    ax.axhline(0.01, color="black", lw=1.0, ls="--", label="Nivel teorico 1%")
    ax.set_title("Tasa de violación anual por modelo (2015-2024)")
    ax.set_xlabel("Año")
    ax.set_ylabel("Tasa de violación")
    ax.set_xticks(sorted(yearly["year"].unique()))
    ax.grid(alpha=0.25)
    ax.legend(ncol=3, fontsize=9)
    savefig("10_violation_rate_lines")


def plot_metric_comparison() -> None:
    summary = pd.read_csv(TABLES / "model_comparison_2015_2024.csv")
    summary["Model"] = summary["Model"].replace({MLP_LONG_LABEL: MLP_LABEL})
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    metrics = ["Violation Rate", "Tick Loss", "Mean VaR Level"]
    valid_models = {"MLP-VaR", "GARCH-t", "CAViaR-AS"}
    colors = ["#1f4e79" if model in valid_models else "#c7c7c7" for model in summary["Model"]]
    for ax, metric in zip(axes, metrics):
        ax.bar(summary["Model"], summary[metric], color=colors)
        ax.set_title(metric)
        ax.tick_params(axis="x", rotation=45)
    fig.text(
        0.5,
        0.01,
        "Barras grises: modelos estadísticamente inválidos (fallan UC o CC)",
        ha="center",
        fontsize=9,
    )
    savefig("08_metric_comparison")


## Generacion de figuras finales

Esta celda ejecuta todas las funciones y guarda cada figura en formatos `.png`, `.pdf` y `.jpg`.


In [ ]:
plot_returns_time_series()
plot_loss_histogram()
plot_qq_normal()
plot_mlp_var_full()
plot_covid_zoom()
plot_benchmark_panel()
plot_violations_by_year()
plot_temporal_violations_valid_models()
plot_violation_rate_lines()
plot_metric_comparison()
